<a href="https://colab.research.google.com/github/pngy87/-PTDLNC-GOOGLE-COLAB/blob/main/BASLINE_DOANNHOM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import gdown
import time

# Tải dữ liệu
file_id = '17Emm31vo3KDA3q4Djxp-0CKiClCAliPG'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'flights.csv'

# Chỉ tải nếu file chưa tồn tại để tiết kiệm thời gian
import os
if not os.path.exists(output):
    gdown.download(url, output, quiet=False)

# Khởi tạo biến df_flights quan trọng
df_flights = pd.read_csv(output, low_memory=False)
print(f"Đã nạp dữ liệu thành công. Kích thước: {df_flights.shape}")

In [ ]:
# 1. Import toàn bộ thư viện cần thiết
import pandas as pd
import numpy as np
import time
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from pandas.tseries.holiday import USFederalHolidayCalendar

# 2. Khởi tạo Scaler
scaler = RobustScaler()

# 3. Định nghĩa các hàm lấy dữ liệu (Data Generators)
def get_orig():
    return scaler.fit_transform(X_train_f), y_train_t, None

def get_weight():
    w = y_train_t.map({0: 1, 1: (len(y_train_t) - sum(y_train_t)) / sum(y_train_t)})
    return scaler.fit_transform(X_train_f), y_train_t, w

def get_down():
    d = pd.concat([X_train_f, y_train_t], axis=1)
    d_min = d[d['IS_DELAYED'] == 1]
    d_maj = d[d['IS_DELAYED'] == 0].sample(n=len(d_min), random_state=42)
    d_bal = pd.concat([d_maj, d_min])
    return scaler.fit_transform(d_bal.drop(columns=['IS_DELAYED'])), d_bal['IS_DELAYED'], None

# 4. Định nghĩa danh sách mô hình
models = {
    "Logistic": LogisticRegression(max_iter=1000),
    "DecisionTree": DecisionTreeClassifier(max_depth=10),
    "RandomForest": RandomForestClassifier(n_estimators=10, max_depth=10, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=100, random_state=42, verbosity=-1),
    "XGBoost": XGBClassifier(n_estimators=100, eval_metric='logloss')
}

print("✅ Đã import thư viện và khởi tạo Models/Scaler thành công!")

In [ ]:
# 1. Chuẩn hóa tên cột (Viết hoa và xóa khoảng trắng thừa)
df_flights.columns = df_flights.columns.str.upper().str.strip()

# 2. Tiền xử lý rút gọn
# Lọc bỏ chuyến bay không thực hiện
df_active = df_flights[df_flights.get(['CANCELLED', 'DIVERTED'], 0).sum(axis=1) == 0].copy()

# Tạo cột DATE nhanh
df_active['DATE'] = pd.to_datetime({'year': 2015, 'month': df_active['MONTH'], 'day': df_active['DAY']})

# Feature Engineering
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start='2015-01-01', end='2015-12-31')
df_active['IS_WEEKEND'] = (df_active['DATE'].dt.weekday >= 5).astype(int)
df_active['IS_HOLIDAY'] = df_active['DATE'].isin(holidays).astype(int)

# Tạo nhãn và loại bỏ cột dư thừa/gây rò rỉ dữ liệu
df_real = df_active.copy()
df_real['IS_DELAYED'] = (df_active['ARRIVAL_DELAY'] > 15).astype(int)

# Danh sách loại bỏ (chỉ bỏ những cột tồn tại)
cols_to_drop = [
    'YEAR', 'FLIGHT_NUMBER', 'TAIL_NUMBER', 'CANCELLATION_REASON', 'DATE',
    'DEPARTURE_DELAY', 'DEPARTURE_TIME', 'WHEELS_OFF', 'WHEELS_ON',
    'ARRIVAL_TIME', 'AIR_TIME', 'TAXI_IN', 'TAXI_OUT', 'ARRIVAL_DELAY',
    'ACTUAL_ELAPSED_TIME', 'ELAPSED_TIME', 'CANCELLED', 'DIVERTED'
]
# Thêm các cột delay chi tiết nếu có
cols_to_drop += [c for c in df_real.columns if 'DELAY' in c and c != 'IS_DELAYED']

df_real = df_real.drop(columns=[c for c in set(cols_to_drop) if c in df_real.columns])

print(f"✅ Xử lý xong! Kích thước: {df_real.shape}")
print(f"Các biến dự báo: {df_real.columns.tolist()}")

In [ ]:
from sklearn.model_selection import train_test_split

# Chia tập Train/Test
X_temp = df_real.drop(columns=['IS_DELAYED'])
y_temp = df_real['IS_DELAYED']
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

# Target Encoding cho Sân bay
global_mean = y_train_t.mean()
for col in ['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']:
    m = y_train_t.groupby(X_train_t[col]).mean()
    X_train_t[f'{col}_EN'] = X_train_t[col].map(m).fillna(global_mean)
    X_test_t[f'{col}_EN'] = X_test_t[col].map(m).fillna(global_mean)

# One-Hot Encoding cho hãng bay và lấp đầy NaN
X_train_f = pd.get_dummies(X_train_t.drop(columns=['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']), columns=['AIRLINE'], drop_first=True)
X_test_f = pd.get_dummies(X_test_t.drop(columns=['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']), columns=['AIRLINE'], drop_first=True)
X_train_f, X_test_f = X_train_f.align(X_test_f, join='left', axis=1, fill_value=0)

# Xử lý NaN triệt để
X_train_f = X_train_f.fillna(X_train_f.median())
X_test_f = X_test_f.fillna(X_train_f.median())

print("Hoàn thành Feature Engineering.")

In [ ]:
# Đảm bảo bạn đã có X_train_f, y_train_t, và X_test_f từ các bước trước

def get_orig():
    """Trả về dữ liệu nguyên bản đã chuẩn hóa"""
    return scaler.fit_transform(X_train_f), y_train_t, None

def get_weight():
    """Tính toán và trả về trọng số lớp để xử lý mất cân bằng"""
    # Trọng số tỷ lệ nghịch với tần suất xuất hiện của lớp
    w = y_train_t.map({
        0: 1,
        1: (len(y_train_t) - sum(y_train_t)) / sum(y_train_t)
    })
    return scaler.fit_transform(X_train_f), y_train_t, w

def get_down():
    """Thực hiện Downsampling để cân bằng tỷ lệ 50/50"""
    d = pd.concat([X_train_f, y_train_t], axis=1)
    d_min = d[d['IS_DELAYED'] == 1]
    # Lấy mẫu ngẫu nhiên lớp đa số bằng với số lượng lớp thiểu số
    d_maj = d[d['IS_DELAYED'] == 0].sample(n=len(d_min), random_state=42)
    d_bal = pd.concat([d_maj, d_min])
    return scaler.fit_transform(d_bal.drop(columns=['IS_DELAYED'])), d_bal['IS_DELAYED'], None

print("Đã định nghĩa xong các hàm: get_orig, get_weight, get_down")

In [ ]:
from sklearn.preprocessing import RobustScaler

# 1. Khởi tạo Scaler - CỰC KỲ QUAN TRỌNG
scaler = RobustScaler()

# 2. Định nghĩa các hàm lấy dữ liệu (Data Generators)
def get_orig():
    """Trả về dữ liệu nguyên bản đã chuẩn hóa"""
    return scaler.fit_transform(X_train_f), y_train_t, None

def get_weight():
    """Tính toán trọng số lớp để xử lý mất cân bằng"""
    w = y_train_t.map({
        0: 1,
        1: (len(y_train_t) - sum(y_train_t)) / sum(y_train_t)
    })
    return scaler.fit_transform(X_train_f), y_train_t, w

def get_down():
    """Thực hiện Downsampling để cân bằng tỷ lệ 50/50"""
    d = pd.concat([X_train_f, y_train_t], axis=1)
    d_min = d[d['IS_DELAYED'] == 1]
    d_maj = d[d['IS_DELAYED'] == 0].sample(n=len(d_min), random_state=42)
    d_bal = pd.concat([d_maj, d_min])
    return scaler.fit_transform(d_bal.drop(columns=['IS_DELAYED'])), d_bal['IS_DELAYED'], None

# 3. Định nghĩa lại danh sách mô hình (để tránh mất biến models)
models = {
    "Logistic": LogisticRegression(max_iter=1000),
    "DecisionTree": DecisionTreeClassifier(max_depth=10),
    "RandomForest": RandomForestClassifier(n_estimators=10, max_depth=10, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=100, random_state=42, verbosity=-1),
    "XGBoost": XGBClassifier(n_estimators=100, eval_metric='logloss')
}

print("✅ Đã khởi tạo Scaler, Models và các hàm get_orig, get_weight, get_down.")

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix

# 1. Khởi tạo Scaler
scaler = RobustScaler()

# 2. Chuẩn hóa tập Test (Phải chạy dòng này để có X_test_scaled)
# Giả sử bạn đã có X_test_f từ bước Feature Engineering
X_test_scaled = scaler.fit_transform(X_test_f)

# 3. Định nghĩa các hàm lấy dữ liệu
def get_orig():
    return scaler.fit_transform(X_train_f), y_train_t, None

def get_weight():
    w = y_train_t.map({0: 1, 1: (len(y_train_t) - sum(y_train_t)) / sum(y_train_t)})
    return scaler.fit_transform(X_train_f), y_train_t, w

def get_down():
    d = pd.concat([X_train_f, y_train_t], axis=1)
    d_min = d[d['IS_DELAYED'] == 1]
    d_maj = d[d['IS_DELAYED'] == 0].sample(n=len(d_min), random_state=42)
    d_bal = pd.concat([d_maj, d_min])
    return scaler.fit_transform(d_bal.drop(columns=['IS_DELAYED'])), d_bal['IS_DELAYED'], None

# 4. Danh sách mô hình
models_dict = {
    "Logistic": LogisticRegression(max_iter=1000),
    "DecisionTree": DecisionTreeClassifier(max_depth=10),
    "RandomForest": RandomForestClassifier(n_estimators=10, max_depth=10, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=100, random_state=42, verbosity=-1),
    "XGBoost": XGBClassifier(n_estimators=100, eval_metric='logloss')
}

print("✅ Đã khởi tạo X_test_scaled và các hàm hỗ trợ.")

In [ ]:
from sklearn.metrics import precision_score, confusion_matrix

results_original = []
print("--- Đang huấn luyện kịch bản: ORIGINAL ---")
X_tr, y_tr, _ = get_orig()

for m_name, m_obj in models.items():
    start = time.time()

    # Huấn luyện mô hình
    m_obj.fit(X_tr, y_tr)

    # Dự đoán nhãn và xác suất
    y_pred = m_obj.predict(X_test_scaled)
    y_prob = m_obj.predict_proba(X_test_scaled)[:, 1] if hasattr(m_obj, "predict_proba") else y_pred

    # Tính toán Confusion Matrix để tìm TN, FP
    tn, fp, fn, tp = confusion_matrix(y_test_t, y_pred).ravel()

    # Tính Specificity (Khả năng đoán đúng các ca KHÔNG TRỄ)
    specificity = tn / (tn + fp)

    results_original.append({
        "Method": "Original",
        "Model": m_name,
        "Accuracy": round(accuracy_score(y_test_t, y_pred), 4),
        "Precision": round(precision_score(y_test_t, y_pred), 4), # Độ chính xác khi báo trễ
        "Recall": round(recall_score(y_test_t, y_pred), 4),       # Khả năng bắt ca trễ
        "F1": round(f1_score(y_test_t, y_pred), 4),               # Cân bằng P & R
        "Specificity": round(specificity, 4),                     # Độ đặc hiệu
        "AUC": round(roc_auc_score(y_test_t, y_prob), 4),         # Khả năng phân loại tổng thể
        "Time(s)": round(time.time() - start, 2)
    })
    print(f"Hoàn thành {m_name} trong {time.time()-start:.2f}s")

# Hiển thị bảng kết quả
df_original = pd.DataFrame(results_original)
display(df_original)

In [ ]:
from sklearn.metrics import precision_score, confusion_matrix

results_weighting = []
print("--- Đang huấn luyện kịch bản: WEIGHTING ---")
X_tr, y_tr, w_tr = get_weight()

for m_name, m_obj in models.items():
    start_time = time.time()

    # Huấn luyện mô hình với sample_weight
    # Lưu ý: w_tr đã được tính toán dựa trên tỷ lệ nghịch của các lớp dữ liệu
    m_obj.fit(X_tr, y_tr, sample_weight=w_tr)

    # Dự đoán nhãn và xác suất
    y_pred = m_obj.predict(X_test_scaled)
    y_prob = m_obj.predict_proba(X_test_scaled)[:, 1] if hasattr(m_obj, "predict_proba") else y_pred

    # Tính toán Confusion Matrix để tìm TN, FP
    tn, fp, fn, tp = confusion_matrix(y_test_t, y_pred).ravel()

    # Tính Specificity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    results_weighting.append({
        "Method": "Weighting",
        "Model": m_name,
        "Accuracy": round(accuracy_score(y_test_t, y_pred), 4),
        "Precision": round(precision_score(y_test_t, y_pred), 4),
        "Recall": round(recall_score(y_test_t, y_pred), 4),
        "F1": round(f1_score(y_test_t, y_pred), 4),
        "Specificity": round(specificity, 4),
        "AUC": round(roc_auc_score(y_test_t, y_prob), 4),
        "Time(s)": round(time.time() - start_time, 2)
    })
    print(f"Hoàn thành {m_name} trong {time.time() - start_time:.2f}s")

# Hiển thị bảng kết quả kịch bản Weighting
df_weighting = pd.DataFrame(results_weighting)
display(df_weighting)

In [ ]:
from sklearn.metrics import precision_score, confusion_matrix

results_downsampling = []
print("--- Đang huấn luyện kịch bản: DOWNSAMPLING ---")
X_tr, y_tr, _ = get_down()

for m_name, m_obj in models.items():
    start_time = time.time()

    # Huấn luyện mô hình trên tập dữ liệu đã cân bằng (Downsampled)
    m_obj.fit(X_tr, y_tr)

    # Dự đoán trên tập Test nguyên bản (để kiểm tra khả năng tổng quát hóa)
    y_pred = m_obj.predict(X_test_scaled)
    y_prob = m_obj.predict_proba(X_test_scaled)[:, 1] if hasattr(m_obj, "predict_proba") else y_pred

    # Tính toán Confusion Matrix để tìm TN, FP
    tn, fp, fn, tp = confusion_matrix(y_test_t, y_pred).ravel()

    # Tính Specificity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    results_downsampling.append({
        "Method": "Downsampling",
        "Model": m_name,
        "Accuracy": round(accuracy_score(y_test_t, y_pred), 4),
        "Precision": round(precision_score(y_test_t, y_pred), 4),
        "Recall": round(recall_score(y_test_t, y_pred), 4),
        "F1": round(f1_score(y_test_t, y_pred), 4),
        "Specificity": round(specificity, 4),
        "AUC": round(roc_auc_score(y_test_t, y_prob), 4),
        "Time(s)": round(time.time() - start_time, 2)
    })
    print(f"Hoàn thành {m_name} trong {time.time() - start_time:.2f}s")

# Hiển thị bảng kết quả kịch bản Downsampling
df_downsampling = pd.DataFrame(results_downsampling)
display(df_downsampling)

In [ ]:
# 1. Gộp toàn bộ kết quả từ 3 biến danh sách đã lưu (giữ nguyên thứ tự kịch bản)
df_final_all = pd.concat([df_original, df_weighting, df_downsampling], ignore_index=True)

# 2. Định dạng hiển thị để dễ quan sát
pd.options.display.float_format = '{:.4f}'.format

print("==========================================================================================")
print("                       BẢNG TỔNG HỢP KẾT QUẢ 15 KỊCH BẢN PHÂN LOẠI                        ")
print("==========================================================================================")

# Hiển thị bảng theo thứ tự gộp (Original -> Weighting -> Downsampling)
display(df_final_all)

# 3. Phân tích để tìm ra mô hình tốt nhất từ bảng tổng hợp
# Tìm kịch bản có F1 cao nhất
best_f1 = df_final_all.loc[df_final_all['F1'].idxmax()]
# Tìm kịch bản có Recall cao nhất
best_recall = df_final_all.loc[df_final_all['Recall'].idxmax()]

print("\n--- PHÂN TÍCH NHANH ---")
print(f"🥇 Mô hình cân bằng nhất (F1-Score): {best_f1['Model']} sử dụng {best_f1['Method']} (F1: {best_f1['F1']:.4f})")
print(f"🚀 Mô hình bắt ca trễ tốt nhất (Recall): {best_recall['Model']} sử dụng {best_recall['Method']} (Recall: {best_recall['Recall']:.4f})")

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_curve, f1_score
from sklearn.preprocessing import RobustScaler

# --- BƯỚC FIX LỖI: KHỞI TẠO LẠI SCALER VÀ TẠO BIẾN SCALED ---
# Đảm bảo bạn đã chạy cell Preprocessing có X_train_f và X_test_f ở trên
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_f)
X_test_scaled = scaler.transform(X_test_f)

# --- BƯỚC 1: CẤU HÌNH MÔ HÌNH TUNING ---
# Tính toán trọng số để xử lý mất cân bằng dữ liệu
counter = y_train_t.value_counts()
scale_weight = counter[0] / counter[1]

best_xgb = XGBClassifier(
    n_estimators=300,        # Tăng số lượng cây
    learning_rate=0.03,      # Tốc độ học nhỏ để ổn định
    max_depth=10,            # Độ sâu lớn để bắt tương quan phức tạp
    scale_pos_weight=scale_weight, # Trọng tâm kỹ thuật Weighting
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',      # Tối ưu tốc độ xử lý cho tập dữ liệu lớn
    random_state=42
)

# --- BƯỚC 2: HUẤN LUYỆN ---
print("🚀 Đang huấn luyện mô hình XGBoost (Tuned)...")
best_xgb.fit(X_train_scaled, y_train_t)

# --- BƯỚC 3: TỐI ƯU HÓA NGƯỠNG (THRESHOLD MOVING) ---
# Lấy xác suất dự báo thay vì nhãn mặc định
y_probs = best_xgb.predict_proba(X_test_scaled)[:, 1]

# Tìm ngưỡng tối ưu để cực đại hóa F1-Score
precisions, recalls, thresholds = precision_recall_curve(y_test_t, y_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
optimal_idx = np.nanargmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"\n✅ Hoàn tất Tuning!")
print(f"🥇 Ngưỡng dự báo tối ưu: {optimal_threshold:.4f}")
print(f"🚀 F1-Score cao nhất đạt được: {f1_scores[optimal_idx]:.4f}")

# --- BƯỚC 4: ĐÁNH GIÁ NHANH ---
y_pred_final = (y_probs >= optimal_threshold).astype(int)
print(f"Accuracy tại ngưỡng mới: {accuracy_score(y_test_t, y_pred_final):.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# --- BƯỚC 1: TỰ ĐỘNG ĐỊNH NGHĨA BIẾN (FIX LỖI NAMEERROR) ---
# Lấy xác suất dự báo từ mô hình đã huấn luyện
y_probs = best_xgb.predict_proba(X_test_scaled)[:, 1]

# Sử dụng ngưỡng tối ưu (optimal_threshold) đã tìm được từ cell trước
# Nếu cell trước chưa chạy, mặc định dùng ngưỡng tối ưu của bạn là 0.5417
try:
    threshold_to_use = optimal_threshold
except NameError:
    threshold_to_use = 0.5417

# Tạo nhãn dự báo tối ưu
y_pred_optimal = (y_probs >= threshold_to_use).astype(int)

# --- BƯỚC 2: IN BÁO CÁO PHÂN LOẠI CHI TIẾT ---
print("==========================================================")
print(f"   KẾT QUẢ MÔ HÌNH TỐI ƯU (Ngưỡng: {threshold_to_use:.4f})")
print("==========================================================")
print(f"Accuracy: {accuracy_score(y_test_t, y_pred_optimal):.4f}")
print(f"F1-Score: {f1_score(y_test_t, y_pred_optimal):.4f}")
print("\nBáo cáo chi tiết:")
print(classification_report(y_test_t, y_pred_optimal))

# --- BƯỚC 3: TRỰC QUAN HÓA MA TRẬN NHẦM LẪN ---
cm = confusion_matrix(y_test_t, y_pred_optimal)
plt.figure(figsize=(8, 6))

# Vẽ Heatmap với nhãn Đúng giờ/Trễ
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Đúng giờ (0)', 'Trễ (1)'],
            yticklabels=['Đúng giờ (0)', 'Trễ (1)'])

plt.title(f'Confusion Matrix (Threshold = {threshold_to_use:.2f})')
plt.xlabel('Dự đoán (Predicted)')
plt.ylabel('Thực tế (Actual)')
plt.show()

In [ ]:
import shap

# 1. Khởi tạo SHAP Explainer cho mô hình XGBoost
# Chúng ta dùng trực tiếp mô hình best_xgb đã huấn luyện
explainer = shap.TreeExplainer(best_xgb)

# 2. Tính toán giá trị SHAP cho tập Test (lấy khoảng 1000 mẫu để vẽ nhanh)
# Nếu dữ liệu quá lớn, lấy mẫu giúp biểu đồ không bị rối và chạy nhanh hơn
X_sample = X_test_f.iloc[:1000]
X_sample_scaled = scaler.transform(X_sample) # Phải dùng dữ liệu đã scale
shap_values = explainer.shap_values(X_test_scaled[:1000])

# Tính toán base value (giá trị trung bình đầu ra của mô hình)
expected_value = explainer.expected_value

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_scaled[:1000], feature_names=X_train_f.columns, show=False)
plt.title("SHAP Summary Plot: Tầm quan trọng của các đặc trưng", fontsize=15)
plt.show()

In [ ]:
# 3. Tạo đối tượng Explanation để vẽ Waterfall
# Lấy quan sát đầu tiên (index = 0)
exp = shap.Explanation(
    values=shap_values[0],
    base_values=expected_value,
    data=X_test_f.iloc[0], # Dùng dữ liệu gốc để hiển thị giá trị thực tế trên biểu đồ
    feature_names=X_train_f.columns
)

plt.figure(figsize=(10, 6))
shap.plots.waterfall(exp)

In [ ]:
# --- KIỂM TRA SỐ LƯỢNG NGÀY LỄ ---
# 1. Đếm tổng số chuyến bay rơi vào ngày lễ (IS_HOLIDAY = 1)
holiday_counts = df_active['IS_HOLIDAY'].value_counts()
total_holiday_flights = holiday_counts.get(1, 0)

# 2. Lấy danh sách các ngày cụ thể được coi là Holiday trong dataset
actual_holidays = df_active[df_active['IS_HOLIDAY'] == 1]['DATE'].unique()
actual_holidays = sorted(actual_holidays)

print("==========================================================")
print(f"📊 THỐNG KÊ BIẾN HOLIDAY")
print("==========================================================")
print(f"📅 Số lượng ngày lễ duy nhất được tìm thấy: {len(actual_holidays)} ngày")
print(f"✈️ Tổng số chuyến bay cất cánh vào ngày lễ: {total_holiday_flights:,} chuyến")
print(f"📈 Tỷ lệ chuyến bay ngày lễ: {(total_holiday_flights/len(df_active))*100:.2f}%")

print("\n📋 Danh sách các ngày lễ cụ thể trong năm 2015:")
for d in actual_holidays:
    # Chuyển sang định dạng YYYY-MM-DD để dễ đọc
    print(f"   - {pd.to_datetime(d).strftime('%Y-%m-%d')}")

In [ ]:
# In danh sách các feature mà mô hình đã học
print("📋 Danh sách các biến độc lập trong mô hình:")
for i, col in enumerate(X_train_f.columns):
    print(f"{i+1}. {col}")

## **DESCRIPTIVE ANALYTICS**

### **1. Tổng quan dataset & biến mục tiêu**

In [ ]:
print("📊 TỔNG QUAN DỮ LIỆU")
print("--------------------------------------------------")
print(f"Số dòng: {df_real.shape[0]:,}")
print(f"Số cột: {df_real.shape[1]}")
print("\nTỷ lệ biến mục tiêu (IS_DELAYED):")
display(df_real['IS_DELAYED'].value_counts(normalize=True).rename("Ratio"))


### **2. Thống kê mô tả các biến số**

In [ ]:
print("📈 THỐNG KÊ MÔ TẢ CÁC BIẾN SỐ")
display(df_real.describe().T)

### **3. Phân phối chuyến bay theo thời gian**

Theo tháng

In [ ]:
flights_by_month = df_active.groupby('MONTH').size()

flights_by_month.plot(
    kind='bar',
    title='Số lượng chuyến bay theo tháng',
    figsize=(10,5)
)

Tỷ lệ trễ theo tháng

In [ ]:
delay_rate_month = df_real.groupby('MONTH')['IS_DELAYED'].mean()

delay_rate_month.plot(
    kind='line',
    marker='o',
    title='Tỷ lệ trễ chuyến theo tháng',
    figsize=(10,5)
)

### **4. Ngày thường vs Cuối tuần**

In [ ]:
weekend_summary = df_real.groupby('IS_WEEKEND')['IS_DELAYED'].agg(
    Total_Flights='count',
    Delay_Rate='mean'
)

display(weekend_summary)

### **5. Ngày lễ vs Ngày thường**

In [ ]:
holiday_summary = df_real.groupby('IS_HOLIDAY')['IS_DELAYED'].agg(
    Total_Flights='count',
    Delay_Rate='mean'
)

display(holiday_summary)


### **6. Phân tích theo hãng hàng không**

In [ ]:
airline_summary = (
    df_real
    .groupby('AIRLINE')['IS_DELAYED']
    .agg(
        Total_Flights='count',
        Delay_Rate='mean'
    )
    .sort_values('Delay_Rate', ascending=False)
)

display(airline_summary.head(10))


### **7. Cuối tuần ảnh hưởng đến hãng**

In [ ]:
airline_weekend_effect = (
    df_real
    .groupby(['AIRLINE', 'IS_WEEKEND'])['IS_DELAYED']
    .mean()
    .unstack()
)

airline_weekend_effect['Weekend_Effect'] = (
    airline_weekend_effect[1] - airline_weekend_effect[0]
)

display(airline_weekend_effect.sort_values('Weekend_Effect', ascending=False))


### **8. Phân tích theo sân bay (Top bận nhất)**

Sân bay xuất phát

In [ ]:
origin_summary = (
    df_real
    .groupby('ORIGIN_AIRPORT')['IS_DELAYED']
    .agg(
        Total_Flights='count',
        Delay_Rate='mean'
    )
    .query("Total_Flights > 1000")
    .sort_values('Delay_Rate', ascending=False)
)

display(origin_summary.head(10))


### **9. Phân tích rủi ro trễ không phụ thuộc lưu lượng**

In [ ]:
airport_risk = (
    df_real
    .groupby('ORIGIN_AIRPORT')['IS_DELAYED']
    .agg(
        Total_Flights='count',
        Delay_Rate='mean'
    )
    .query("Total_Flights > 1000")
)

display(airport_risk.sort_values('Delay_Rate', ascending=False))

### **10. Correlation giữa các biến số**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

num_cols = [
    'MONTH', 'DAY', 'IS_WEEKEND', 'IS_HOLIDAY', 'IS_DELAYED'
]

plt.figure(figsize=(8,6))
sns.heatmap(
    df_real[num_cols].corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)
plt.title("Correlation Matrix (Descriptive Analytics)")
plt.show()